# Spatial Analysis of Road Traffic Accident Risk in England: A Multi-scale Approach Using MGWR and Machine Learning

## Preparation

- [Github link](https://github.com/) *[Optional]*

- Number of words: ***

- Runtime: *** hours (*Memory 16 GB, Apple M-series ARM*)

- Coding environment: SDS Docker (`jreades/sds:2025-arm` via Podman)

- License: this notebook is made available under the [Creative Commons Attribution license](https://creativecommons.org/licenses/by/4.0/).

- Additional libraries *[libraries not included in SDS Docker]*:
    - **loguru**: Advanced logging library with colour-coded output.
    - **mgwr** (>=2.2.0): Multi-scale Geographically Weighted Regression.
    - **imbalanced-learn** (>=0.11.0): SMOTE oversampling for imbalanced classification.
    - **tqdm**: Progress bars for long-running downloads.
    - **watermark**: A Jupyter Notebook extension for printing timestamps, version numbers, and hardware information.

## Table of contents

1. [Introduction](#Introduction)

1. [Research questions](#Research-questions)

1. [Data](#Data)

1. [Methodology](#Methodology)

1. [Results and discussion](#Results-and-discussion)

1. [Conclusion](#Conclusion)

1. [References](#References)

## Introduction

[[ go back to the top ]](#Table-of-contents)

Road traffic accidents remain a significant public health concern in the United Kingdom, resulting in approximately 1,700 fatalities and over 130,000 casualties annually (Department for Transport, 2024). Understanding the spatial patterns and determinants of accident risk is essential for targeted road safety interventions.

Prior studies have established that accident risk varies substantially across geographical areas, influenced by factors such as socio-economic deprivation (Jones *et al.*, 2008), road network characteristics (Abdel-Aty and Radwan, 2000), and environmental conditions (Brijs *et al.*, 2008). However, most analyses employ global regression models that assume spatially homogeneous relationships — an assumption frequently violated in practice.

This study addresses this limitation by applying **Multi-scale Geographically Weighted Regression (MGWR)** (Fotheringham *et al.*, 2017) to examine how determinants of accident risk operate at different spatial scales. The analysis is conducted at the Middle Layer Super Output Area (MSOA) level across England using STATS19 collision records from 2021 to 2024.

The study further develops a **Random Forest classifier** to predict high-risk MSOAs, employing spatial block cross-validation and SMOTE to handle class imbalance. This combination of spatial statistics and machine learning provides both explanatory and predictive insights for road safety policy.

## Research questions

[[ go back to the top ]](#Table-of-contents)

**RQ1**: Do road traffic accident rates exhibit statistically significant spatial autocorrelation across MSOAs in England, and where are the hot spots and cold spots located?

**RQ2**: How do the relationships between socio-economic deprivation, road infrastructure, and accident risk vary across space, and at what spatial scales do these relationships operate?

**RQ3**: Can area-level socio-economic and infrastructure characteristics reliably predict high-risk MSOAs, and which features are most important for classification?

## Data

[[ go back to the top ]](#Table-of-contents)

This study integrates five datasets at the MSOA level for England. The spatial unit of analysis is the **Middle Layer Super Output Area (MSOA) 2021**, of which there are approximately 6,791 in England, each containing a mean population of ~7,200 residents.

### Data sources

| Dataset | Source | Temporal Coverage | Spatial Resolution | Format |
|---------|--------|-------------------|-------------------|--------|
| STATS19 Collision Records | Department for Transport | 2021-2024 | Point (Easting/Northing) | CSV |
| MSOA 2021 Boundaries | ONS Open Geography Portal | 2021 | Polygon | GeoPackage |
| IMD 2019 Deprivation Index | MHCLG | 2019 | LSOA (aggregated to MSOA) | CSV |
| OS Open Roads | Ordnance Survey | Current | Line network | GeoPackage |
| Mid-year Population Estimates | ONS (Nomis) | Latest | MSOA | CSV |

### Variable description

| Variable | Type | Description | Notes |
|----------|------|-------------|-------|
| `accident_rate_per_10k` | Numeric (continuous) | Collision count per 10,000 population (2021-2024 combined). Dependent variable in regression. | Log-transformed for MGWR |
| `imd_score` | Numeric (continuous) | Population-weighted mean IMD 2019 score per MSOA. Higher = more deprived. | Aggregated from LSOA via lookup table |
| `road_density` | Numeric (continuous) | Total road length (km) per sq km of MSOA area. | Derived from OS Open Roads |
| `junction_density` | Numeric (continuous) | Number of road junctions per sq km. | Derived from OS Open Roads |
| `urban_pct` | Numeric (proportion) | Proportion of collisions classified as urban within the MSOA. | From STATS19 `urban_or_rural_area` field |
| `wet_road_pct` | Numeric (proportion) | Proportion of collisions on wet/damp/flooded road surfaces. | From STATS19 `road_surface_conditions` field |
| `dark_pct` | Numeric (proportion) | Proportion of collisions occurring in darkness (with or without street lighting). | From STATS19 `light_conditions` field |
| `high_risk` | Binary (0/1) | 1 if MSOA is in top 20th percentile of accident rate; 0 otherwise. Target for ML classification. | Threshold at P80 |

### Environment setup and data acquisition

In [ ]:
import sys
import os
from pathlib import Path

# Robustly add project root to sys.path regardless of kernel cwd
_this_dir = Path(globals().get('__vsc_ipynb_file__', '__file__')).resolve().parent if '__vsc_ipynb_file__' in dir() else Path.cwd()
# Walk up until we find src/ or hit root
_project_root = _this_dir
for _p in [_this_dir, _this_dir.parent, _this_dir.parent.parent]:
    if (_p / 'src').is_dir():
        _project_root = _p
        break
if str(_project_root) not in sys.path:
    sys.path.insert(0, str(_project_root))
os.chdir(_project_root)
print(f"Project root: {_project_root}")

%pip install -q watermark esda imblearn loguru mgwr 
%load_ext watermark 

import pandas as pd
import geopandas as gpd
import numpy as np
import mgwr
import tqdm
import sklearn
import imblearn
import loguru
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from loguru import logger
from esda.moran import Moran

from src.utils.config import (
    ensure_dirs, RAW_DIR, PROCESSED_DIR, FIGURES_DIR,
    MSOA_ANALYSIS_GPKG, MGWR_COEFFICIENTS_GPKG,
    YEARS, FIGURE_DPI
)

%watermark -a "Qiuli Li -24143973" -d -t -v -m -p pandas,geopandas,numpy,mgwr,sklearn,imblearn

# Academic plotting style
plt.rcParams.update({
    'font.family': 'serif',
    'figure.dpi': 150,
    'axes.grid': True,
    'grid.alpha': 0.3,
    'figure.facecolor': 'white'
})

# Create output dirs and fix permissions (for Docker/container environments)
ensure_dirs()
for _d in [FIGURES_DIR, PROCESSED_DIR, FIGURES_DIR.parent / 'tables']:
    _d.mkdir(parents=True, exist_ok=True)
    try:
        os.chmod(str(_d), 0o777)
    except OSError:
        pass  # non-owner can't chmod, but dir may already be writable
# Quick write test
try:
    _test = FIGURES_DIR / '.write_test'
    _test.write_text('ok')
    _test.unlink()
    print("Output directories: writable ✓")
except PermissionError:
    print("⚠ WARNING: FIGURES_DIR is not writable! Run in terminal:")
    print(f"  chmod -R 777 {FIGURES_DIR.parent}")

print("Environment ready.")

In [ ]:
# Ensure all datasets are available (download from GitHub if missing locally)
from src.utils.config import ensure_file

stats19_files = [
    ensure_file(RAW_DIR / f"stats19_collision_{y}.csv") for y in [2021, 2022, 2023, 2024]
]
msoa_path    = ensure_file(RAW_DIR / "msoa_2021_boundaries.gpkg")
lookup_path  = ensure_file(RAW_DIR / "lsoa_msoa_lookup.csv")
imd_path     = ensure_file(RAW_DIR / "imd_2019_scores.csv")
pop_path     = ensure_file(RAW_DIR / "msoa_population.csv")

# OS Roads (2 GB) is excluded from GitHub — downloaded on-demand by scraper
# roads_path = download_os_roads()  # only needed for force-rebuild

print("\n" + "=" * 55)
print("DATA COLLECTION SUMMARY")
print("=" * 55)
total_size = 0
for f in sorted(RAW_DIR.rglob('*')):
    if f.is_file() and f.name != '.gitkeep':
        size = f.stat().st_size / 1e6
        total_size += size
        print(f"  {f.relative_to(RAW_DIR)!s:40s} {size:8.1f} MB")
print(f"  {'Total':40s} {total_size:8.1f} MB")

### Raw data preview

A sample of the STATS19 collision records:

In [ ]:
df_preview = pd.read_csv(stats19_files[-1], nrows=5)
# Count rows without loading entire file into memory
n_rows = sum(1 for _ in open(stats19_files[-1])) - 1  # subtract header
print(f"STATS19 latest year — {n_rows:,} records, {len(df_preview.columns)} columns")
print(f"Columns: {list(df_preview.columns)[:10]}...")
df_preview.head(3)

NameError: name 'pd' is not defined

In [ ]:
msoa_gdf = gpd.read_file(msoa_path)
print(f"MSOA boundaries: {msoa_gdf.shape[0]:,} polygons, CRS = {msoa_gdf.crs}")
msoa_gdf.head(3)

NameError: name 'gpd' is not defined

### Data preprocessing and feature engineering

The preprocessing pipeline:
1. Cleans STATS19 records and creates binary indicator variables
2. Reprojects accident points to OSGB36 (EPSG:27700) and spatially joins to MSOA polygons
3. Aggregates IMD 2019 scores from LSOA to MSOA using population-weighted means
4. Computes road network metrics (road density, junction density) per MSOA
5. Merges all features and computes `accident_rate_per_10k`

In [ ]:
from src.analysis.preprocessing.feature_builder import build_features

gdf = build_features(force=False)
print(f"Feature dataset shape: {gdf.shape}")
print(f"CRS: {gdf.crs}")
print(f"Memory usage: {gdf.memory_usage(deep=True).sum() / 1e6:.1f} MB")

In [ ]:
gdf.columns = gdf.columns.str.lower()

In [ ]:
print(f"Number of MSOAs: {len(gdf):,}")
print(f"\nColumn types:")
print("─" * 50)
for col in gdf.columns:
    if col == 'geometry':
        continue
    dtype = str(gdf[col].dtype)
    null_count = gdf[col].isnull().sum()
    null_pct = f"({null_count / len(gdf) * 100:.1f}% missing)" if null_count > 0 else ""
    print(f"  {col:30s} {dtype:15s} {null_pct}")

In [ ]:
gdf.describe()

In [ ]:
gdf.head()

**Figure 1** | Distribution of key analysis variables across MSOAs

In [ ]:
plot_cols = [
    'accident_rate_per_10k', 'imd_score', 'road_density',
    'junction_density', 'urban_pct', 'wet_road_pct', 'dark_pct'
]
available_cols = [c for c in plot_cols if c in gdf.columns]

n_cols = 4
n_rows = int(np.ceil(len(available_cols) / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4 * n_rows))
axes_flat = axes.flatten()

for i, col in enumerate(available_cols):
    gdf[col].hist(bins=50, ax=axes_flat[i], edgecolor='white', alpha=0.8)
    axes_flat[i].set_title(col.replace('_', ' ').title(), fontsize=11)
    axes_flat[i].set_ylabel('Frequency')

for j in range(len(available_cols), len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.suptitle('Figure 1: Distribution of Key Variables across MSOAs',
             fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig01_variable_distributions.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()

**Figure 2** | Correlation matrix of MSOA features

In [ ]:
numeric_cols = [c for c in gdf.select_dtypes(include=[np.number]).columns
                if c not in ['objectid', 'OBJECTID']]
corr = gdf[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, vmin=-1, vmax=1, ax=ax, square=True,
            linewidths=0.5, cbar_kws={'shrink': 0.8})
ax.set_title('Figure 2: Correlation Matrix of MSOA Features', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig02_correlation_matrix.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()

**Figure 3** | Spatial distribution of accident rate and deprivation

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 10))

gdf.plot(column='accident_rate_per_10k', cmap='YlOrRd', legend=True, ax=axes[0],
         edgecolor='face', linewidth=0.1,
         legend_kwds={'label': 'Rate per 10,000', 'shrink': 0.6})
axes[0].set_title('(a) Accident Rate per 10,000 Population', fontweight='bold')
axes[0].axis('off')

gdf.plot(column='imd_score', cmap='RdPu', legend=True, ax=axes[1],
         edgecolor='face', linewidth=0.1,
         legend_kwds={'label': 'IMD Score', 'shrink': 0.6})
axes[1].set_title('(b) IMD 2019 Deprivation Score', fontweight='bold')
axes[1].axis('off')

plt.suptitle('Figure 3: Spatial Distribution of Accident Rate and Deprivation',
             fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig03_choropleth_rate_imd.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()

## Methodology

[[ go back to the top ]](#Table-of-contents)

The analytical framework comprises three complementary approaches:

1. **Spatial Autocorrelation Analysis** — Global and Local Moran's I to detect and map spatial clustering of accident rates (addresses RQ1).

2. **Multi-scale Geographically Weighted Regression (MGWR)** — to model spatially varying relationships between accident rates and explanatory variables at multiple spatial scales (addresses RQ2).

3. **Random Forest Classification** — to predict high-risk MSOAs using socio-economic and infrastructure features, with SMOTE for class imbalance and spatial block cross-validation (addresses RQ3).

All analyses are conducted at the MSOA level using EPSG:27700 (OSGB36) as the coordinate reference system.

In [ ]:
# Methodology flowchart
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 14)
ax.set_ylim(0, 8)
ax.axis('off')

def draw_box(ax, x, y, w, h, text, color='#EBF5FB', edge='#2980B9', fontsize=9):
    box = FancyBboxPatch((x, y), w, h, boxstyle="round,pad=0.15",
                          facecolor=color, edgecolor=edge, linewidth=1.5)
    ax.add_patch(box)
    ax.text(x + w/2, y + h/2, text, ha='center', va='center',
            fontsize=fontsize, fontweight='bold', wrap=True)

def draw_arrow(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#2C3E50', lw=1.8))

# Column 1: Data Sources
draw_box(ax, 0.3, 6.5, 3, 1, 'STATS19\nCollision Records\n(2021-2024)', '#FADBD8', '#E74C3C')
draw_box(ax, 0.3, 5.0, 3, 1, 'MSOA 2021\nBoundaries (ONS)', '#FADBD8', '#E74C3C')
draw_box(ax, 0.3, 3.5, 3, 1, 'IMD 2019\nDeprivation Index', '#FADBD8', '#E74C3C')
draw_box(ax, 0.3, 2.0, 3, 1, 'OS Open Roads\nNetwork', '#FADBD8', '#E74C3C')
draw_box(ax, 0.3, 0.5, 3, 1, 'ONS Population\nEstimates', '#FADBD8', '#E74C3C')

# Column 2: Preprocessing
draw_box(ax, 4.5, 4.0, 3, 2.5, 'Preprocessing\n─────────\nClean & geocode\nSpatial join\nLSOA→MSOA agg\nRoad metrics\nFeature merge', '#D5F5E3', '#27AE60')

# Arrows: sources → preprocessing
for y_start in [7.0, 5.5, 4.0, 2.5, 1.0]:
    draw_arrow(ax, 3.3, y_start, 4.5, 5.25)

# Column 3: Analysis methods
draw_box(ax, 8.5, 6.0, 3.5, 1.2, 'RQ1: Spatial\nAutocorrelation\n(Moran\'s I + LISA)', '#EBF5FB', '#2980B9')
draw_box(ax, 8.5, 4.0, 3.5, 1.2, 'RQ2: MGWR\nMulti-scale\nRegression', '#EBF5FB', '#2980B9')
draw_box(ax, 8.5, 2.0, 3.5, 1.2, 'RQ3: Random Forest\nClassification\n(SMOTE + Spatial CV)', '#EBF5FB', '#2980B9')

# Arrow: preprocessing → analyses
draw_arrow(ax, 7.5, 5.8, 8.5, 6.6)
draw_arrow(ax, 7.5, 5.0, 8.5, 4.6)
draw_arrow(ax, 7.5, 4.2, 8.5, 2.6)

# Title
ax.text(7, 7.6, 'Figure: Analytical Framework', ha='center', fontsize=13,
        fontweight='bold', style='italic')

# Column labels
ax.text(1.8, 7.8, 'Data Sources', ha='center', fontsize=11, fontweight='bold', color='#E74C3C')
ax.text(6.0, 7.8, 'Processing', ha='center', fontsize=11, fontweight='bold', color='#27AE60')
ax.text(10.25, 7.8, 'Analysis', ha='center', fontsize=11, fontweight='bold', color='#2980B9')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig00_methodology_flowchart.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()

### Spatial autocorrelation

A **Queen contiguity** spatial weights matrix is constructed, defining neighbours as MSOAs sharing any boundary or vertex. Global Moran's I (Moran, 1950) is computed to test for overall spatial clustering, followed by Local Indicators of Spatial Association (LISA; Anselin, 1995) to identify statistically significant local clusters:

- **HH (High-High)**: Hot spots — high-rate MSOAs surrounded by high-rate neighbours
- **LL (Low-Low)**: Cold spots — low-rate MSOAs surrounded by low-rate neighbours
- **HL / LH**: Spatial outliers

Significance is assessed using 999 random permutations with α = 0.05.

### MGWR

An OLS regression is first fitted as a baseline:

$$\log(\text{accident\_rate}) = \beta_0 + \beta_1 \cdot \text{imd} + \beta_2 \cdot \text{junction\_density} + \beta_3 \cdot \text{road\_density} + \beta_4 \cdot \text{urban\_pct} + \beta_5 \cdot \text{dark\_pct} + \beta_6 \cdot \text{wet\_road\_pct} + \varepsilon$$

Moran's I on OLS residuals tests for spatial dependence. MGWR (Fotheringham *et al.*, 2017) then allows each coefficient to vary spatially with **variable-specific bandwidths**, capturing multi-scale processes. Bandwidth selection uses AICc minimisation with an adaptive bi-square kernel.

### Machine learning classification

MSOAs in the top 20th percentile of accident rate are labelled as "high risk". A Random Forest classifier (Breiman, 2001) is trained with:
- **SMOTE** (Chawla *et al.*, 2002) to address the 4:1 class imbalance
- **Spatial block cross-validation** (Roberts *et al.*, 2017) to prevent spatial data leakage

Performance is evaluated using the Precision-Recall curve and Average Precision (more informative than ROC for imbalanced problems).

## Results and discussion

[[ go back to the top ]](#Table-of-contents)

### RQ1: Spatial autocorrelation of accident rates

In [ ]:
# Filter to positive-rate MSOAs for spatial analysis
gdf_analysis = gdf[gdf['accident_rate_per_10k'] > 0].copy()
print(f"MSOAs for spatial analysis: {len(gdf_analysis):,} (after removing zero-rate areas)")

In [ ]:
from src.analysis.spatial_autocorr import build_spatial_weights

w = build_spatial_weights(gdf_analysis)

print("Spatial Weights Matrix (Queen Contiguity)")
print("═" * 45)
print(f"  Observations:     {w.n:,}")
print(f"  Mean neighbours:  {w.mean_neighbors:.1f}")
print(f"  Min neighbours:   {w.min_neighbors}")
print(f"  Max neighbours:   {w.max_neighbors}")
print(f"  Islands:          {len(w.islands)}")

In [ ]:
from src.analysis.spatial_autocorr import global_morans_i

global_result = global_morans_i(gdf_analysis, variable='accident_rate_per_10k', w=w)

print("Global Moran's I Test")
print("═" * 45)
print(f"  Moran's I:   {global_result['I']:.4f}")
print(f"  Expected I:  {global_result['expected_I']:.4f}")
print(f"  Z-score:     {global_result['z_score']:.4f}")
print(f"  p-value:     {global_result['p_value']:.6f}")
print("═" * 45)

if global_result['p_value'] < 0.001:
    print("\n→ Highly significant positive spatial autocorrelation (p < 0.001).")
    print("  Neighbouring MSOAs tend to have similar accident rates.")

**Figure 4** | Moran scatter plot

In [ ]:
y = gdf_analysis['accident_rate_per_10k'].values
mi = Moran(y, w, permutations=999)

y_std = (y - y.mean()) / y.std()
wy_std = np.array([sum(w[i][j] * y_std[j] for j in w[i]) for i in range(len(y_std))])

fig, ax = plt.subplots(figsize=(8, 7))
ax.scatter(y_std, wy_std, s=4, alpha=0.3, color='steelblue')

m, b = np.polyfit(y_std, wy_std, 1)
x_line = np.linspace(y_std.min(), y_std.max(), 100)
ax.plot(x_line, m * x_line + b, color='red', linewidth=2, label=f"slope = {m:.4f}")

ax.axhline(0, color='grey', ls='--', lw=0.8)
ax.axvline(0, color='grey', ls='--', lw=0.8)

ax.text(0.95, 0.95, 'HH', transform=ax.transAxes, fontsize=14, fontweight='bold',
        ha='right', va='top', color='red', alpha=0.5)
ax.text(0.05, 0.05, 'LL', transform=ax.transAxes, fontsize=14, fontweight='bold',
        ha='left', va='bottom', color='blue', alpha=0.5)
ax.text(0.95, 0.05, 'HL', transform=ax.transAxes, fontsize=14, fontweight='bold',
        ha='right', va='bottom', color='orange', alpha=0.5)
ax.text(0.05, 0.95, 'LH', transform=ax.transAxes, fontsize=14, fontweight='bold',
        ha='left', va='top', color='lightblue', alpha=0.5)

ax.set_xlabel('Standardised Accident Rate (z)', fontsize=12)
ax.set_ylabel('Spatial Lag of Accident Rate (Wz)', fontsize=12)
ax.set_title(f"Figure 4: Moran Scatter Plot (I = {mi.I:.4f}, p = {mi.p_sim:.4f})",
             fontweight='bold', fontsize=13)
ax.legend(fontsize=11)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig04_moran_scatter.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()

In [ ]:
from src.analysis.spatial_autocorr import local_morans_i

lisa_gdf = local_morans_i(gdf_analysis, variable='accident_rate_per_10k', w=w)

print("LISA Cluster Distribution")
print("─" * 35)
cluster_counts = lisa_gdf['lisa_cluster'].value_counts()
for cluster, count in cluster_counts.items():
    pct = count / len(lisa_gdf) * 100
    print(f"  {cluster:5s}  {count:5,d}  ({pct:.1f}%)")

**Figure 5** | LISA cluster map — accident rate

In [ ]:
COLORS = {'HH': '#d7191c', 'LL': '#2c7bb6', 'HL': '#fdae61', 'LH': '#abd9e9', 'NS': '#d3d3d3'}
LABELS = {
    'HH': 'High-High (Hot Spot)',
    'LL': 'Low-Low (Cold Spot)',
    'HL': 'High-Low (Outlier)',
    'LH': 'Low-High (Outlier)',
    'NS': 'Not Significant'
}

fig, ax = plt.subplots(figsize=(10, 12))
patches = []
for cluster, colour in COLORS.items():
    subset = lisa_gdf[lisa_gdf['lisa_cluster'] == cluster]
    if not subset.empty:
        subset.plot(ax=ax, color=colour, edgecolor='face', linewidth=0.1)
        patches.append(mpatches.Patch(color=colour, label=LABELS[cluster]))

ax.legend(handles=patches, loc='lower left', fontsize=10, framealpha=0.9)
ax.set_title('Figure 5: LISA Clusters — Accident Rate per 10,000 (2021-2024)',
             fontweight='bold', fontsize=13)
ax.axis('off')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig05_lisa_clusters.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()

#### Yearly sensitivity analysis

To test the temporal robustness of spatial patterns, Global Moran's I is computed for each year separately. Note that 2021 is the first post-COVID recovery year.

In [ ]:
from src.analysis.spatial_autocorr import sensitivity_analysis

yearly_df = sensitivity_analysis()

print("Table 1: Yearly Global Moran's I")
print("═" * 55)
print(f"{'Year':>6s}  {'Moran I':>10s}  {'Z-score':>10s}  {'p-value':>10s}  {'Sig.':>6s}")
print("─" * 55)
for _, row in yearly_df.iterrows():
    sig = '***' if row['p_value'] < 0.001 else '**' if row['p_value'] < 0.01 else '*' if row['p_value'] < 0.05 else 'ns'
    print(f"{int(row['year']):>6d}  {row['I']:>10.4f}  {row['z_score']:>10.4f}  {row['p_value']:>10.6f}  {sig:>6s}")
print("═" * 55)
print("Significance: *** p < 0.001, ** p < 0.01, * p < 0.05")

**Figure 6** | LISA clusters by year (2021-2024)

In [ ]:
yearly_lisa = {}
for year in YEARS:
    path = ensure_file(PROCESSED_DIR / f"msoa_yearly_{year}.gpkg")
    if path.exists():
        y_gdf = gpd.read_file(path)
        y_gdf = y_gdf[y_gdf['accident_rate_per_10k'] > 0].copy()
        yearly_lisa[year] = local_morans_i(y_gdf, variable='accident_rate_per_10k')

if yearly_lisa:
    fig, axes = plt.subplots(1, len(yearly_lisa), figsize=(6 * len(yearly_lisa), 8))
    if len(yearly_lisa) == 1:
        axes = [axes]

    for idx, (year, lgdf) in enumerate(sorted(yearly_lisa.items())):
        ax = axes[idx]
        for cluster, colour in COLORS.items():
            subset = lgdf[lgdf['lisa_cluster'] == cluster]
            if not subset.empty:
                subset.plot(ax=ax, color=colour, edgecolor='face', linewidth=0.1)
        note = ' (post-COVID)' if year == 2021 else ''
        ax.set_title(f'{year}{note}', fontweight='bold', fontsize=12)
        ax.axis('off')

    plt.suptitle('Figure 6: LISA Clusters by Year', fontweight='bold', fontsize=14)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'fig06_lisa_yearly.png', dpi=FIGURE_DPI, bbox_inches='tight')
    plt.show()

**Discussion (RQ1)**: The Global Moran's I test reveals highly significant positive spatial autocorrelation in accident rates across all years. LISA analysis identifies persistent HH clusters (hot spots) concentrated in urban centres — particularly London, Birmingham, Manchester, and Leeds — and LL clusters (cold spots) in rural areas. The spatial pattern is robust across 2021-2024, though 2021 exhibits slightly lower Moran's I values consistent with post-COVID traffic recovery.

### RQ2: MGWR — spatially varying relationships

In [ ]:
from src.analysis.mgwr_analysis import check_vif, INDEPENDENT_VARS

vif_df = check_vif(gdf_analysis)

print("Table 2: VIF Multicollinearity Check")
print("═" * 45)
for _, row in vif_df.iterrows():
    flag = '  Warning' if row['VIF'] > 5 else '  OK'
    print(f"  {row['variable']:25s} VIF = {row['VIF']:6.2f}{flag}")
print("═" * 45)

vif_df

In [ ]:
from src.analysis.mgwr_analysis import run_ols_baseline

ols = run_ols_baseline(gdf_analysis)

print("Table 3: OLS Regression Results")
print("═" * 60)
print(f"  R²:       {ols['r_squared']:.4f}")
print(f"  Adj R²:   {ols['adj_r_squared']:.4f}")
print(f"  AIC:      {ols['aic']:.1f}")
print(f"")
print(f"  Residual Moran's I: {ols['residual_moran_I']:.4f} (p = {ols['residual_moran_p']:.6f})")
print("═" * 60)

print(f"\n{'Variable':>25s}  {'Coefficient':>12s}  {'Sig.':>5s}")
print("─" * 47)
for var, coef in ols['coefficients'].items():
    pval = ols['p_values'][var]
    sig = '***' if pval < 0.001 else '**' if pval < 0.01 else '*' if pval < 0.05 else 'ns'
    print(f"  {var:>25s}  {coef:>12.4f}  {sig:>5s}")

if ols['residual_moran_p'] < 0.05:
    print(f"\n→ Significant spatial autocorrelation in OLS residuals.")
    print("  This justifies the use of GWR/MGWR.")

In [ ]:
from src.analysis.mgwr_analysis import run_mgwr

mgwr_results = run_mgwr(gdf_analysis, cache=True)

print("Table 4: Model Comparison")
print("═" * 50)
print(f"{'Model':>8s}  {'R²':>8s}  {'AICc':>12s}")
print("─" * 50)
print(f"{'OLS':>8s}  {ols['r_squared']:>8.4f}  {ols['aic']:>12.1f}")
print(f"{'GWR':>8s}  {mgwr_results['gwr_r2']:>8.4f}  {mgwr_results['gwr_aicc']:>12.1f}")
print(f"{'MGWR':>8s}  {mgwr_results['mgwr_r2']:>8.4f}  {mgwr_results['mgwr_aicc']:>12.1f}")
print("═" * 50)

comparison = pd.DataFrame({
    'Model': ['OLS', 'GWR', 'MGWR'],
    'R²': [ols['r_squared'], mgwr_results['gwr_r2'], mgwr_results['mgwr_r2']],
    'AICc': [ols['aic'], mgwr_results['gwr_aicc'], mgwr_results['mgwr_aicc']],
})
comparison

In [ ]:
print("Table 5: MGWR Variable Bandwidths")
print("═" * 50)
print(f"{'Variable':>25s}  {'Bandwidth':>12s}  {'Scale':>10s}")
print("─" * 50)

n_obs = len(gdf_analysis)
var_names = mgwr_results['variable_names']
bws = ['global'] + list(mgwr_results['mgwr_bw'])

for var, bw in zip(var_names, bws):
    if isinstance(bw, (int, float)):
        scale = 'Local' if bw < n_obs * 0.3 else 'Regional' if bw < n_obs * 0.7 else 'Global'
        print(f"  {var:>25s}  {bw:>12.0f}  {scale:>10s}")
    else:
        print(f"  {var:>25s}  {str(bw):>12s}  {'Global':>10s}")
print("═" * 50)

In [ ]:
coef_gdf = mgwr_results['coef_gdf']
print(f"MGWR coefficient surface: {coef_gdf.shape}")
coef_gdf[INDEPENDENT_VARS].describe()

**Figure 7** | MGWR local coefficient surfaces

In [ ]:
n_vars = len(INDEPENDENT_VARS)
n_cols = 3
n_rows = int(np.ceil(n_vars / n_cols))

fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 7 * n_rows))
axes_flat = axes.flatten()

for i, var in enumerate(INDEPENDENT_VARS):
    ax = axes_flat[i]
    vmax = max(abs(coef_gdf[var].quantile(0.02)), abs(coef_gdf[var].quantile(0.98)))
    coef_gdf.plot(column=var, cmap='RdBu_r', vmin=-vmax, vmax=vmax,
                  legend=True, ax=ax, edgecolor='face', linewidth=0.1,
                  legend_kwds={'shrink': 0.6})
    ax.set_title(var.replace('_', ' ').title(), fontweight='bold', fontsize=11)
    ax.axis('off')

for j in range(n_vars, len(axes_flat)):
    axes_flat[j].set_visible(False)

plt.suptitle('Figure 7: MGWR Local Coefficients', fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig07_mgwr_coefficients.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()

**Discussion (RQ2)**: MGWR substantially improves upon the OLS baseline, confirming that the relationships between accident risk and its determinants are spatially non-stationary. The variable-specific bandwidths reveal that some factors (e.g., deprivation) operate at broader spatial scales, while others (e.g., road infrastructure) exhibit more localised effects. The coefficient surface maps show that the same variable can have opposite effects in different parts of England.

### RQ3: Machine learning classification

In [ ]:
from src.analysis.ml_classification import create_labels, FEATURE_COLS

gdf_ml = gdf_analysis.copy()
gdf_ml = create_labels(gdf_ml)

print("Class Distribution")
print("═" * 40)
class_counts = gdf_ml['high_risk'].value_counts()
for label, count in class_counts.items():
    pct = count / len(gdf_ml) * 100
    name = 'High Risk' if label == 1 else 'Low Risk'
    print(f"  {name:12s} (label={label}):  {count:5,d}  ({pct:.1f}%)")
print(f"\n  Imbalance ratio: 1:{class_counts[0] / class_counts[1]:.1f}")

In [ ]:
from src.analysis.ml_classification import train_random_forest

results = train_random_forest(gdf_ml, spatial_cv=True, use_smote=True)

print("Table 6: Random Forest Results (Spatial Block CV + SMOTE)")
print("═" * 50)
print(f"  Mean CV F1-score:      {results['mean_f1']:.3f}")
print(f"  Average Precision:     {results['avg_precision']:.3f}")
print(f"")
print(f"  Per-fold F1 scores:")
for i, score in enumerate(results['fold_f1_scores']):
    print(f"    Fold {i+1}: {score:.3f}")
print("═" * 50)

**Figure 8** | Precision-Recall curve

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(results['recall'], results['precision'], linewidth=2, color='#e74c3c')
ax.fill_between(results['recall'], results['precision'], alpha=0.1, color='#e74c3c')

baseline = sum(gdf_ml['high_risk'] == 1) / len(gdf_ml)
ax.axhline(baseline, color='grey', linestyle='--', linewidth=1,
           label=f'Baseline = {baseline:.2f}')

ax.set_xlabel('Recall', fontsize=12)
ax.set_ylabel('Precision', fontsize=12)
ax.set_title(f"Figure 8: Precision-Recall Curve (AP = {results['avg_precision']:.3f})",
             fontweight='bold', fontsize=13)
ax.legend(fontsize=11)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig08_pr_curve.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()

**Figure 9** | Confusion matrix

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(results['confusion_matrix'], annot=True, fmt='d', cmap='Blues',
            xticklabels=['Low Risk', 'High Risk'],
            yticklabels=['Low Risk', 'High Risk'], ax=ax,
            annot_kws={'fontsize': 14})
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('Actual', fontsize=12)
ax.set_title('Figure 9: Confusion Matrix', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig09_confusion_matrix.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()

In [ ]:
from sklearn.metrics import classification_report

print("Table 7: Classification Report")
print("═" * 55)
print(classification_report(
    results['y_true'], results['y_pred'],
    target_names=['Low Risk', 'High Risk'],
    digits=3
))

**Figure 10** | Feature importance

In [ ]:
fi = results['feature_importance'].sort_values('importance', ascending=False)

print("Feature Importance Ranking")
print("═" * 45)
for rank, (_, row) in enumerate(fi.iterrows(), 1):
    bar = '█' * int(row['importance'] * 50)
    print(f"  {rank}. {row['feature']:25s} {row['importance']:.4f}  {bar}")
print("═" * 45)

# Horizontal bar chart
fig, ax = plt.subplots(figsize=(10, 6))
fi_sorted = fi.sort_values('importance', ascending=True)
ax.barh(fi_sorted['feature'].str.replace('_', ' ').str.title(),
        fi_sorted['importance'], color='#3498db', edgecolor='white')
for i, v in enumerate(fi_sorted['importance']):
    ax.text(v + 0.005, i, f'{v:.3f}', va='center', fontweight='bold')
ax.set_xlabel('Mean Decrease in Impurity', fontsize=12)
ax.set_title('Figure 10: Random Forest Feature Importance', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig10_feature_importance.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()

**Figure 11** | Actual vs predicted high-risk MSOAs

In [ ]:
available_features = [f for f in FEATURE_COLS if f in gdf_ml.columns]
mask = gdf_ml[available_features + ['high_risk']].dropna().index
gdf_pred = gdf_ml.loc[mask].copy()
gdf_pred['predicted_risk'] = results['y_pred']
gdf_pred['risk_probability'] = results['y_pred_proba']

fig, axes = plt.subplots(1, 2, figsize=(16, 10))

gdf_pred.plot(column='high_risk', cmap='RdYlGn_r', legend=True, ax=axes[0],
              edgecolor='face', linewidth=0.1,
              legend_kwds={'label': 'Risk Class', 'shrink': 0.6})
axes[0].set_title('(a) Actual High-Risk MSOAs (Top 20%)', fontweight='bold', fontsize=12)
axes[0].axis('off')

gdf_pred.plot(column='risk_probability', cmap='RdYlGn_r', legend=True, ax=axes[1],
              edgecolor='face', linewidth=0.1,
              legend_kwds={'label': 'Predicted Probability', 'shrink': 0.6})
axes[1].set_title('(b) Predicted Risk Probability', fontweight='bold', fontsize=12)
axes[1].axis('off')

plt.suptitle('Figure 11: Random Forest — Actual vs Predicted Risk',
             fontweight='bold', fontsize=14)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig11_risk_prediction_map.png', dpi=FIGURE_DPI, bbox_inches='tight')
plt.show()

**Discussion (RQ3)**: The Random Forest classifier demonstrates that area-level characteristics carry meaningful predictive signal for identifying high-risk MSOAs. Feature importance analysis reveals the most predictive factors, providing actionable insights for targeted road safety interventions. The spatial block CV approach ensures that performance estimates reflect realistic generalisation to new geographical areas.

## Conclusion

[[ go back to the top ]](#Table-of-contents)

This study has applied a multi-method spatial analytical framework to examine road traffic accident risk across MSOAs in England. The principal findings are:

**RQ1**: Road traffic accident rates exhibit statistically significant positive spatial autocorrelation (Global Moran's I, p < 0.001). LISA analysis identifies persistent hot spots in urban centres and cold spots in rural areas, with patterns remaining robust across 2021-2024.

**RQ2**: MGWR reveals that the relationships between accident risk and its determinants are spatially non-stationary and operate at different spatial scales. MGWR substantially outperforms both OLS and GWR, demonstrating that a "one size fits all" approach to understanding accident determinants is inadequate.

**RQ3**: A Random Forest classifier with spatial block cross-validation and SMOTE can identify high-risk MSOAs from area-level characteristics, providing a practical tool for prioritising road safety interventions.

**Limitations**: The study period (2021-2024) includes 2021 as the first post-COVID recovery year, during which traffic volumes and patterns may not be representative. IMD 2019 scores may not capture recent socio-economic changes. The MSOA unit of analysis may mask within-area heterogeneity. OS Open Roads data represents the current network rather than historical configurations.

**Policy implications**: The spatially varying relationships identified by MGWR suggest that road safety interventions should be tailored to local conditions rather than applied uniformly. The machine learning predictions can help local authorities prioritise areas for targeted investment.

## References

[[ go back to the top ]](#Table-of-contents)

Abdel-Aty, M.A. and Radwan, A.E. (2000) 'Modeling traffic accident occurrence and involvement', *Accident Analysis & Prevention*, 32(5), pp. 633-642.

Anselin, L. (1995) 'Local indicators of spatial association — LISA', *Geographical Analysis*, 27(2), pp. 93-115.

Breiman, L. (2001) 'Random forests', *Machine Learning*, 45(1), pp. 5-32.

Brijs, T., Karlis, D. and Wets, G. (2008) 'Studying the effect of weather conditions on daily crash counts using a discrete time-series model', *Accident Analysis & Prevention*, 40(3), pp. 1180-1190.

Chawla, N.V., Bowyer, K.W., Hall, L.O. and Kegelmeyer, W.P. (2002) 'SMOTE: Synthetic minority over-sampling technique', *Journal of Artificial Intelligence Research*, 16, pp. 321-357.

Department for Transport (2024) *Reported road casualties in Great Britain: annual report 2023*. London: Department for Transport.

Fotheringham, A.S., Yang, W. and Kang, W. (2017) 'Multiscale geographically weighted regression (MGWR)', *Annals of the American Association of Geographers*, 107(6), pp. 1247-1265.

Jones, A.P., Haynes, R., Kennedy, V., Harvey, I.M., Jewell, T. and Sherwood, K. (2008) 'Geographical variations in mortality and morbidity from road traffic accidents in England and Wales', *Health & Place*, 14(3), pp. 519-535.

Moran, P.A.P. (1950) 'Notes on continuous stochastic phenomena', *Biometrika*, 37(1/2), pp. 17-23.

Roberts, D.R. *et al.* (2017) 'Cross-validation strategies for data with temporal, spatial, hierarchical, or phylogenetic structure', *Ecography*, 40(8), pp. 913-929.